# Week 4 -- Function 8: Neural Network Surrogate + Bayesian Optimisation (8D)

In [ ]:
# -- WEEKLY OBSERVATIONS LOG --------------------------------------------------------------------------------
import numpy as np

INITIAL_SIZE = 40
DATA_PATH_X  = '../data/function_8/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_8/initial_outputs.npy'

weekly_log = [
    ([0.816815, 0.656459, 0.258882, 0.854757, 0.737459, 0.233756, 0.838932, 0.820863], 7.275150),  # Week 1
    ([0.856208, 0.914280, 0.313812, 0.369507, 0.710367, 0.257166, 0.646775, 0.025767], 7.627456),  # Week 2 -- improved
    ([0.076274, 0.101214, 0.383035, 0.338493, 0.113685, 0.882235, 0.615428, 0.796463], 9.038246),  # Week 3 EI -- IMPROVED
    # Week 4: add next week's result here as ([x1..x8], y_value)
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE + 1
print(f'Week {current_week} | {len(Y_disk)} total observations ({INITIAL_SIZE} initial + {len(Y_disk)-INITIAL_SIZE} added)')

In [ ]:
# Cell 3: Load data and inspect
# Function 8: Complex ML hyperparameter tuning (8D), Maximise

X = np.load('../data/function_8/initial_inputs.npy')
Y = np.load('../data/function_8/initial_outputs.npy')

print(f'Input  shape : {X.shape}   (n_samples x n_dimensions)')
print(f'Output shape : {Y.shape}  (n_samples,)')
print()

pairs = sorted(zip(Y.tolist(), X.tolist()), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 104)
print('  Top 10 observations (sorted descending by Y)')
print('=' * 104)
for i, (y_val, x_val) in enumerate(zip(Y_sorted[:10], X_sorted[:10])):
    marker = '  <-- best' if i == 0 else ''
    xs = ', '.join([f'{v:.5f}' for v in x_val])
    print(f'  [{i+1:2d}]  X = [{xs}]   Y = {y_val:+.4f}{marker}')
print(f'  ... ({len(Y_sorted) - 10} more observations)')
print('=' * 104)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
n_dims = X.shape[1]
print(f'\n  Best Y*  = {best_Y:.4f}')
xs_str = ', '.join([f'{v:.8f}' for v in best_X])
print(f'  Best X*  = [{xs_str}]')

In [ ]:
# Cell 4: Fit GP
import warnings
warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

Y_fit = np.log(np.abs(Y) + 1e-300)

kernel = RBF(length_scale=0.1, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10)
gp.fit(X, Y_fit)

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                   : {gp.kernel_}')
print(f'  Log-marginal-likelihood  : {gp.log_marginal_likelihood_value_:.4f}')
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
actual_log = np.log(np.abs(best_Y) + 1e-300)
print(f'  Sanity check at best X*  : GP mean={pred_mean[0]:.4f}  actual={actual_log:.4f}  std={pred_std[0]:.6f}')
print('=' * 62)

In [ ]:
# Cell 5: Neural Network Surrogate Model (TensorFlow/Keras)
# F8 (8D): wider network (128 units, 2000 epochs) for higher-dimensional input.
# Following course Assignment 15.1 (Sequential API) + Assignment 15.2 (GradientTape training).

import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42)

Y_log    = np.log(np.abs(Y) + 1e-300)
y_scaler = StandardScaler()
Y_scaled = y_scaler.fit_transform(Y_log.reshape(-1, 1)).ravel().astype(np.float32)

X_t = tf.constant(X.astype(np.float32))
Y_t = tf.constant(Y_scaled)

# Wider network for 8D input (Assignment 15.1 style)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu',
                          kernel_regularizer=tf.keras.regularizers.l2(1e-3),
                          input_shape=(n_dims,)),
    tf.keras.layers.Dropout(0.1),
    tf.keras.layers.Dense(128, activation='relu',
                          kernel_regularizer=tf.keras.regularizers.l2(1e-3)),
    tf.keras.layers.Dropout(0.1),
    tf.keras.layers.Dense(1)
], name='surrogate_nn')

# Loss function and optimizer (Assignment 15.2 style)
loss_fn   = tf.keras.losses.MeanSquaredError()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

# train_step using GradientTape (Assignment 15.2 style)
@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        y_pred = tf.squeeze(model(x, training=True))
        loss   = loss_fn(y, y_pred) + tf.add_n(model.losses)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

for epoch in range(2000):
    loss = train_step(X_t, Y_t)
    if epoch % 400 == 0:
        print(f'  Epoch {epoch:4d}  MSE (scaled) = {loss.numpy():.4f}')

print(f'\n  Final training MSE (scaled): {loss.numpy():.4f}')

nn_pred_best = float(tf.squeeze(model(tf.expand_dims(
    tf.constant(best_X.astype(np.float32)), 0), training=False)))
print(f'  NN prediction at best X*: {nn_pred_best:.4f} (scaled)')

In [ ]:
# Cell 6: Gradient-Guided Acquisition with Trust-Region + Multi-Start (TensorFlow)
# Assignment 15.2 style: tf.Variable + tf.GradientTape for each restart.
# F8: improving strongly -- exploit hard with 3 restarts.

import numpy as np

trust_radius = 0.15   # F8: improving -- tight exploitation on best-known seed

restart_configs = [
    (best_X.copy(), trust_radius),                             # exploit: constrained
    (np.random.RandomState(7).uniform(0, 1, n_dims), None),   # explore: unconstrained
    (np.random.RandomState(13).uniform(0, 1, n_dims), None),  # explore: unconstrained
]

best_grad_score = -np.inf
grad_candidate  = best_X.copy()

for seed_idx, (seed_x, radius) in enumerate(restart_configs):
    lo_r = np.clip(seed_x - radius, 0.0, 1.0).astype(np.float32) if radius is not None else np.zeros(n_dims, dtype=np.float32)
    hi_r = np.clip(seed_x + radius, 0.0, 1.0).astype(np.float32) if radius is not None else np.ones(n_dims, dtype=np.float32)

    x_opt     = tf.Variable(seed_x.copy().astype(np.float32))
    opt_inner = tf.keras.optimizers.Adam(learning_rate=5e-3)

    for step in range(800):
        with tf.GradientTape() as tape:
            score      = tf.squeeze(model(tf.expand_dims(x_opt, 0), training=False))
            loss_inner = -score
        gradients = tape.gradient(loss_inner, [x_opt])
        opt_inner.apply_gradients(zip(gradients, [x_opt]))
        x_opt.assign(tf.clip_by_value(x_opt, lo_r, hi_r))

    candidate = x_opt.numpy().copy()
    score_val = float(tf.squeeze(model(tf.expand_dims(
        tf.constant(candidate.astype(np.float32)), 0), training=False)))
    tag = f'trust±{radius}' if radius else 'free'
    xs_str = ','.join([f'{v:.4f}' for v in candidate])
    print(f'  Restart {seed_idx+1} ({tag}): [{xs_str}]  score={score_val:.4f}')
    if score_val > best_grad_score:
        best_grad_score = score_val
        grad_candidate  = candidate

# --- Strategy B: GP UCB (tight exploitation) ---
beta = 1.5
np.random.seed(42)
X_grid = np.random.uniform(0, 1, size=(50000, n_dims))
post_mean, post_std = gp.predict(X_grid, return_std=True)
ucb_scores = post_mean + beta * post_std
gp_candidate = X_grid[np.argmax(ucb_scores)]

# --- Pick winner ---
nn_score_grad = best_grad_score
nn_score_gp   = float(tf.squeeze(model(tf.expand_dims(
    tf.constant(gp_candidate.astype(np.float32)), 0), training=False)))

selected = 'NN gradient ascent' if nn_score_grad >= nn_score_gp else 'GP UCB'
next_x   = grad_candidate if nn_score_grad >= nn_score_gp else gp_candidate

portal_string = '-'.join([f'{v:.6f}' for v in next_x])

print()
print('=' * 104)
print('  Acquisition Results')
print('=' * 104)
grad_str = ', '.join([f'{v:.6f}' for v in grad_candidate])
gp_str   = ', '.join([f'{v:.6f}' for v in gp_candidate])
print(f'  NN gradient candidate : [{grad_str}]  NN score: {nn_score_grad:.4f}')
print(f'  GP UCB  candidate     : [{gp_str}]  NN score: {nn_score_gp:.4f}')
print(f'  Selected method       : {selected}')
print()
next_str = ', '.join([f'{v:.6f}' for v in next_x])
print(f'  Next query point      : [{next_str}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 104)

In [ ]:
# Cell 7: Input Gradient Sensitivity Analysis (TensorFlow)
# Assignment 15.2 style: tf.Variable watched by tf.GradientTape to compute ∂output/∂x_i.
# For 8D, this identifies which dimensions actually drive the output.

x_sens = tf.Variable(best_X.copy().astype(np.float32))

with tf.GradientTape() as tape:
    out = tf.squeeze(model(tf.expand_dims(x_sens, 0), training=False))

grads = np.abs(tape.gradient(out, x_sens).numpy())

xs_str = ', '.join([f'{v:.5f}' for v in best_X])
print('Input gradient magnitudes at best X*:')
print(f'  (best X* = [{xs_str}])')
print(f'  (Y* = {best_Y:.4f})')
print()
max_g = grads.max() if grads.max() > 0 else 1.0
for i, g in enumerate(grads):
    bar = '#' * max(1, int(g * 40 / max_g))
    print(f'  x{i+1}: |∂f/∂x{i+1}| = {g:.4f}  {bar}')

ranked = np.argsort(grads)[::-1]
print(f'\n  Dimension importance ranking (most → least influential):')
print(f'  ' + ' > '.join([f'x{r+1}' for r in ranked]))
print(f'\n  Insight: low-gradient dimensions are near-optimal or irrelevant.')
print(f'  High-gradient dimensions are where future queries should focus.')

In [ ]:
# Cell 7b: SVM Classification Check (secondary validation)
# For F8 (8D), SVM also reveals which dimensions define the good/bad boundary
# through its support vector positions.

from sklearn.svm import SVC

threshold = np.percentile(Y, 70)   # top 30% = 'good'
labels = (Y >= threshold).astype(int)

print(f'  Classification threshold (70th percentile of Y): {threshold:.4f}')
print(f'  Good samples: {labels.sum()} / {len(labels)}')
print()

if labels.sum() >= 2 and (labels == 0).sum() >= 2:
    svm_clf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
    svm_clf.fit(X, labels)

    prob_best  = svm_clf.predict_proba(best_X.reshape(1, -1))[0, 1]
    prob_query = svm_clf.predict_proba(next_x.reshape(1, -1))[0, 1]

    print(f'  SVM support vectors    : {svm_clf.n_support_} (per class)')
    print(f'  P(good) at best X*     : {prob_best:.3f}  (reference)')
    print(f'  P(good) at next query  : {prob_query:.3f}')
    print()
    if prob_query >= 0.5:
        print('  ✓ SVM endorses next query as likely good -- consistent with improvement trend')
    elif prob_query >= 0.3:
        print('  ~ SVM uncertain -- near boundary; query is at the edge of the known good region')
    else:
        print('  ✗ SVM flags query as likely poor -- consider widening trust-region or using a free restart')
else:
    print('  Insufficient class balance for SVM training (need >= 2 samples per class).')

In [ ]:
# Cell 8: Summary
xs_str = ', '.join([f'{v:.6f}' for v in best_X])
next_str = ', '.join([f'{v:.6f}' for v in next_x])
print('=' * 104)
print('  SUMMARY -- Week 4 Bayesian Optimisation (NN Surrogate)')
print('=' * 104)
print(f'  Function             : 8D Complex ML Hyperparameter Tuning')
print(f'  Objective            : Maximise')
print(f'  GP kernel            : RBF(length_scale=0.1, fixed)')
print(f'  NN architecture      : MLP [8 → 128 → 128 → 1], ReLU, Adam lr=1e-3, 2000 epochs')
print(f'  Acquisition          : NN gradient ascent x3 restarts vs GP UCB (beta=1.5, 50k grid)')
print(f'  Selected method      : {selected}')
print()
print(f'  Progress: W1=7.275 → W2=7.627 → W3=9.038  (+19% last week)')
print(f'  Current best X*      : [{xs_str}]')
print(f'  Current best Y*      : {best_Y:.4f}')
print()
print(f'  Next query point     : [{next_str}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 104)

## Week 4 -- Reflection

**Function**: F8 -- Complex ML Hyperparameter Tuning (8D)

### Query
- **Method**: NN gradient ascent with 3 random restarts (wider network: 128 units), with GP UCB (beta=1.5, 50k grid) as fallback.
- **Week 3 lesson**: Jumped from 7.627 to 9.038 (+19%) using EI. The gradient sensitivity analysis will reveal which of the 8 dimensions are actually driving the improvement.
- **Query type**: *(fill in after running -- Exploration or Exploitation?)*
- **Input submitted**: *(fill in portal submission string)*
- **Output received**: *(fill in after receiving result)*
- **Best so far**: *(update after result)*

### What this result taught us
*(fill in after receiving result)*

### Strategy for Week 5
*(fill in after reflecting on result)*